# LLM open-weight trên Colab T4 cho LegacyLens

Notebook này chạy mô hình open-weight Qwen2.5 Instruct (giấy phép Apache-2.0) bằng Ollama trên GPU T4 miễn phí của Colab,
rồi mở một đường hầm Cloudflare để Codespaces gọi tới bằng API kiểu OpenAI.

Làm theo `HUONG_DAN.md`, bước 5. Tóm tắt:

1. **Runtime > Change runtime type > T4 GPU > Save.**
2. **Runtime > Run all.** Lần đầu mất khoảng 5-8 phút (cài Ollama, tải mô hình 4,7 GB).
3. Ô cuối in ra 3 dòng `LLM_...`. Chép vào file `.env` trong Codespaces.
4. Giữ tab này mở trong lúc chạy `make ask`, `make extract`, `make eval-qa`.

Lưu ý an toàn: URL đường hầm không có mật khẩu. Không chia sẻ URL. Khi xong, chọn
**Runtime > Disconnect and delete runtime**.

## 1. Chọn mô hình

Giữ mặc định cho lần chạy đầu. Muốn thử mô hình lớn hơn (bước 8 của hướng dẫn), đổi sang dòng có ghi chú.

In [ ]:
MODEL = "qwen2.5:7b-instruct"  # 4,7 GB, nhanh, đủ cửa sổ 32k trên T4
CONTEXT = 32768

# MODEL = "qwen2.5:14b-instruct"  # 9 GB, chính xác hơn nhưng chậm hơn khoảng 2 lần
# CONTEXT = 16384                 # T4 có 15 GB, mô hình 14B chỉ vừa cửa sổ 16k

## 2. Kiểm tra GPU

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(gpu.stdout or "KHÔNG CÓ GPU. Vào Runtime > Change runtime type > T4 GPU, rồi chạy lại từ đầu.")

## 3. Cài Ollama

In [ ]:
!sudo apt-get -qq update > /dev/null && sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## 4. Khởi động máy chủ Ollama

In [ ]:
import os
import time
import urllib.request

env = os.environ | {
    "OLLAMA_HOST": "0.0.0.0:11434",
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT),  # mặc định của Ollama chỉ 4k, quá ngắn cho agent đọc code
    "OLLAMA_KEEP_ALIVE": "-1",              # giữ mô hình trong GPU, không tải lại sau mỗi 5 phút
    "OLLAMA_NUM_PARALLEL": "1",
}
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        print("Ollama đang chạy:", urllib.request.urlopen("http://localhost:11434/api/version").read().decode())
        break
    except OSError:
        time.sleep(1)
else:
    print("Ollama chưa chạy, xem ollama.log")

## 5. Tải mô hình (vài phút)

In [ ]:
!ollama pull {MODEL}

## 6. Thử gọi tool

LegacyLens cần mô hình gọi được tool. Ô này phải in `Gọi tool OK`.

In [ ]:
import json

req = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Read lines 1-5 of program COSGN00C."}],
    "tools": [{"type": "function", "function": {
        "name": "read_lines",
        "description": "Read source lines of a COBOL program",
        "parameters": {"type": "object", "properties": {
            "program": {"type": "string"}, "start": {"type": "integer"}, "end": {"type": "integer"}},
            "required": ["program", "start", "end"]}}}],
}
t0 = time.time()
r = urllib.request.Request("http://localhost:11434/v1/chat/completions", json.dumps(req).encode(), {"Content-Type": "application/json"})
msg = json.load(urllib.request.urlopen(r))["choices"][0]["message"]
calls = msg.get("tool_calls") or []
print(f"Gọi tool OK ({time.time() - t0:.1f}s):", calls[0]["function"]) if calls else print("Mô hình không gọi tool:", msg.get("content"))

## 7. Mở đường hầm và lấy URL

In [ ]:
import re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
    stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT,
)
url = None
for _ in range(40):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
if url:
    print("Chép 4 dòng dưới vào .env trong Codespaces (thay các dòng LLM_ cũ):\n")
    print(f"LLM_BASE_URL={url}/v1")
    print(f"LLM_MODEL={MODEL}")
    print("LLM_API_KEY=ollama")
    print(f"LLM_CONTEXT={CONTEXT}")
else:
    print("Chưa có URL. Chạy lại ô này. Nếu vẫn lỗi, xem tunnel.log")

## 8. Khi có lỗi

Chạy ô dưới để xem bộ nhớ GPU và 20 dòng log cuối của Ollama.

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!ollama ps
!tail -n 20 ollama.log